# 🚖 Exploratory Data Analysis: NYC Taxi ETAs

This notebook explores the taxi trip dataset to identify key patterns, outliers, and correlations that will inform our feature engineering and modeling process.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import preprocess_pipeline

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# Load data
DATA_PATH = '../data/raw/taxi_trip_data.csv'
# For this notebook, we'll use a sample if the file is large
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")

## 1. Target Distribution

The target variable is `trip_duration`. Understanding its distribution helps us decide if transformations (like log-transform) are necessary.

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df['trip_duration'], bins=50, kde=True, color='teal')
plt.title('Trip Duration Distribution')
plt.xlabel('Duration (seconds)')

plt.subplot(1, 2, 2)
sns.histplot(np.log1p(df['trip_duration']), bins=50, kde=True, color='orange')
plt.title('Log-Transformed Trip Duration')
plt.xlabel('Log(Duration + 1)')

plt.tight_layout()
plt.show()

## 2. Trip Distance vs. Duration

Distance is the strongest predictor of duration. We use the Haversine formula to approximate the actual distance traveled.

In [ ]:
from src.features import haversine_distance

df['distance_km'] = haversine_distance(
    df['pickup_latitude'], df['pickup_longitude'],
    df['dropoff_latitude'], df['dropoff_longitude']
)

plt.figure(figsize=(10, 6))
sns.scatterplot(x='distance_km', y='trip_duration', data=df.sample(min(10000, len(df))), alpha=0.1)
plt.title('Trip Distance vs. Duration')
plt.xlabel('Distance (km)')
plt.ylabel('Duration (s)')
plt.show()

## 3. Time-Based Patterns

ETAs are highly sensitive to traffic cycles. We look at how duration changes by hour of the day and day of the week.

In [ ]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.day_name()

plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
sns.lineplot(x='hour', y='trip_duration', data=df, errorbar='sd', color='blue')
plt.title('Average Trip Duration by Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Avg Duration (s)')

plt.subplot(1, 2, 2)
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sns.barplot(x='day_of_week', y='trip_duration', data=df, order=day_order, palette='viridis')
plt.title('Average Trip Duration by Day')
plt.xticks(rotation=45)
plt.show()

## 4. Outlier Detection

Extreme values (e.g., trips taking 0 seconds or 24 hours) can bias the model. We use boxplots to visualize the range of normal operations.

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(x=df['trip_duration'], color='lightcoral')
plt.title('Boxplot of Trip Duration')
plt.xlabel('Duration (s)')
plt.show()

print(f"Trips < 30s: {len(df[df['trip_duration'] < 30])}")
print(f"Trips > 3 hours: {len(df[df['trip_duration'] > 10800])}")

## 🏁 EDA Insights

1. **Non-Linearity**: Distance and duration are correlated, but the variance increases significantly for longer trips due to traffic.
2. **Seasonality**: Rush hours (8-10 AM, 4-7 PM) show clearly elevated trip durations.
3. **Skewness**: The target variable is right-skewed; a log-transformation during modeling might improve convergence.